# FLIR Dimensionality Reduction — Project Report

Revisión del protocolo completo de reducción dimensional. Cada punto representa un contenido visual exacto único. Las configuraciones seleccionadas son **candidatas exploratorias**; esta etapa no produce asignaciones de agrupamiento.

## 1. Objetivo

Obtener proyecciones reproducibles con t-SNE y PaCMAP, medir qué relaciones preservan y estudiar sensibilidad a hiperparámetros y semillas. Se reutilizan los espacios completos DINOv2 y CLIP verificados; los 1657 registros históricos conservan su mapping hacia 1459 contenidos, sin aumentar artificialmente la densidad por copias exactas.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, Markdown, display

report = Path("reports/reduction")
tables = report / "tables"
metadata = json.loads((report / "report_metadata.json").read_text(encoding="utf-8"))
runs = pd.read_csv(tables / "runs.csv")
configs = pd.read_csv(tables / "configuration_summary.csv")
stability = pd.read_csv(tables / "configuration_stability.csv")
references = pd.read_csv(tables / "candidate_reference_metrics.csv")
parameters = pd.read_csv(tables / "effective_parameters.csv")
def table(frame):
    display(HTML(frame.to_html(index=False, float_format=lambda v: f"{v:.4f}", border=0, escape=True)))
def figure(name):
    display(Image(filename=str(report / "figures" / name)))
display(Markdown(f"**Evidencia ejecutada:** {metadata['run_count']} reducciones completas, N={metadata['N']} por ejecución, {metadata['figure_count']} figuras y cuatro referencias seleccionadas. Todos los artefactos de entrada y las 36 reducciones fueron verificados antes de construir este reporte."))

## 2. Input feature spaces

Entrada exclusiva: `embeddings_l2.npy` completo de cada encoder. Se preservan raw, L2 y sus índices originales. No se concatenan encoders ni se renormaliza la entrada. `original_split`, secuencia, índice temporal, clases, cajas y conteos de objetos no entran al ajuste. La referencia de evaluación reutiliza el coseno y los vecinos top-20 de la fase anterior.

In [ ]:
table(pd.read_csv(tables / "input_spaces.csv"))

## 3. Por qué reducir dimensionalidad

La proyección permite inspeccionar relaciones geométricas y preparar hipótesis contrastables para clustering. Distancias, separaciones y densidades se alteran al proyectar a 2D; una región visualmente separada no demuestra un clúster ni calidad semántica. 2D es la salida principal para interpretación; la API admite 3D, pendiente de un propósito analítico. No se asume que 2D sea la entrada óptima del agrupamiento posterior.

## 4. t-SNE protocol

Perplexity **10, 30 y 50** × semillas **0, 1 y 2**: nueve ejecuciones por encoder. Hasta **1000 iteraciones**, learning rate `auto`, early exaggeration 12, Barnes-Hut angle 0.5, distancia euclidiana e inicialización PCA. La PCA solo inicializa las coordenadas; **no existe pre-reducción PCA de los embeddings**. Se mantienen las 384/512 dimensiones al construir las relaciones. Se controla un hilo nativo y se registra la tasa efectiva de aprendizaje.

Configuraciones genéricas versionadas en `configs/reduction/tsne_research.yaml`; bibliotecas y parámetros efectivos por ejecución están en metadata local.

In [ ]:
table(parameters.loc[parameters.method.eq("tsne"), ["encoder", "label", "seed", "library_version", "learning_rate_effective", "input_dimension", "output_dimension"]])

## 5. PaCMAP protocol

Base controlada: **n_neighbors=10, MN_ratio=0.5, FP_ratio=2**. Variaciones de MN_ratio: **0.2 y 1.0**, conservando los demás parámetros. Cada configuración usa semillas 0/1/2, learning rate 1 y fases **100/100/250** (450 iteraciones). Por contenido se solicitan 10 near pairs, 2/5/10 mid-near pairs y 20 further pairs. Se verifican los conteos efectivos y se conservan las huellas del muestreo.

`apply_pca=False` evita la pre-reducción automática de PaCMAP. Su transformación afín interna resta un mínimo escalar, divide por un rango escalar y centra columnas; preserva distancias euclidianas salvo una escala común positiva y redondeo. La PCA de inicialización no reemplaza la entrada al grafo. FAISS HNSW, sus versiones y la política de un hilo quedan registrados. Las semillas no garantizan igualdad bit a bit en otras plataformas.

Configuración versionada: `configs/reduction/pacmap_research.yaml`.

In [ ]:
table(parameters.loc[parameters.method.eq("pacmap"), ["encoder", "label", "seed", "library", "library_version", "input_dimension", "output_dimension"]].drop_duplicates())

## 6. Preservation metrics

Todas las métricas se calculan contra el espacio original del mismo encoder: distancia **1 − coseno** de vectores L2, contrastada con los vecinos guardados. En 2D se usa distancia euclidiana. Los empates se resuelven por content_id ascendente y se excluye el propio contenido.

- **Trustworthiness@5/10/20 (T):** penaliza vecinos intrusos según su rango original.
- **Continuity@5/10/20 (C):** penaliza vecinos originales omitidos según su rango reducido. Se utiliza la fórmula exacta, probada contra trustworthiness con los espacios invertidos.
- **Jaccard@5/10/20 (J):** intersección/unión de conjuntos de vecinos, por contenido; media, mediana y Q1/Q3.
- **Spearman:** orden de distancias en 100000 pares únicos sin reemplazo, semilla 0 y la misma selección canónica en ambos encoders. Es complementaria; no se calculan p-valores que supongan pares independientes.
- **Estabilidad:** Jaccard entre semillas 0/1, 0/2 y 1/2 para cada k; invariante a rotación, reflexión, traslación y escala uniforme.

T y C usan la normalización exacta 2/[N·k·(2N−3k−1)] y k<N/2. Una T alta puede coexistir con cambios considerables en el conjunto exacto de vecinos; ninguna cifra mide por sí sola la validez semántica.

In [ ]:
table(pd.read_csv(tables / "timings.csv"))
display(Markdown("Tiempos observados en CPU local, no benchmark de hardware. `fit` mide el ajuste; `backend_total` incluye importación/preparación y `evaluation_total` mide las métricas. La verificación y agregación añaden tiempo. Se conserva el tiempo de cada run en las tablas siguientes."))

## 7. DINOv2 results

La primera figura responde cómo cambian preservación y sensibilidad entre configuraciones. Las barras representan desviación estándar entre tres semillas, no incertidumbre inferencial. Las figuras de referencia corresponden a la semilla 0 de las configuraciones seleccionadas por la regla de métricas descrita en la sección 12. Cada proyección tiene su propia escala; no contiene etiquetas de clustering.

In [ ]:
figure("01_reduction_quality_dinov2.png")
current = runs.loc[runs.encoder.eq("dinov2")].copy()
columns = ["method", "label", "seed", "fit_seconds"] + [f"{metric}@{k}" for metric in ("trustworthiness", "continuity") for k in (5, 10, 20)] + ["spearman_distance"]
table(current[columns])
display(Markdown("**Preservación de vecinos por run.** Se muestran media, mediana y cuartiles para los tres tamaños de vecindario; las filas por content_id permanecen en los artefactos locales."))
preservation = []
for row in current.to_dict("records"):
    for k in (5, 10, 20):
        preservation.append({"method": row["method"], "label": row["label"], "seed": row["seed"], "k": k,
                             **{key: row[f"jaccard@{k}_{key}"] for key in ("mean_jaccard", "median_jaccard", "Q1", "Q3")}})
table(pd.DataFrame(preservation))
figure("03_pacmap_dinov2_reference.png")
figure("05_tsne_dinov2_reference.png")

## 8. CLIP results

La primera figura responde cómo cambian preservación y sensibilidad entre configuraciones. Las barras representan desviación estándar entre tres semillas, no incertidumbre inferencial. Las figuras de referencia corresponden a la semilla 0 de las configuraciones seleccionadas por la regla de métricas descrita en la sección 12. Cada proyección tiene su propia escala; no contiene etiquetas de clustering.

In [ ]:
figure("02_reduction_quality_clip.png")
current = runs.loc[runs.encoder.eq("clip")].copy()
columns = ["method", "label", "seed", "fit_seconds"] + [f"{metric}@{k}" for metric in ("trustworthiness", "continuity") for k in (5, 10, 20)] + ["spearman_distance"]
table(current[columns])
display(Markdown("**Preservación de vecinos por run.** Se muestran media, mediana y cuartiles para los tres tamaños de vecindario; las filas por content_id permanecen en los artefactos locales."))
preservation = []
for row in current.to_dict("records"):
    for k in (5, 10, 20):
        preservation.append({"method": row["method"], "label": row["label"], "seed": row["seed"], "k": k,
                             **{key: row[f"jaccard@{k}_{key}"] for key in ("mean_jaccard", "median_jaccard", "Q1", "Q3")}})
table(pd.DataFrame(preservation))
figure("04_pacmap_clip_reference.png")
figure("06_tsne_clip_reference.png")

## 9. Stability across seeds

¿Los mismos contenidos mantienen vecindarios parecidos cuando cambia la semilla? Se comparan conjuntos de vecinos y no posiciones absolutas. Los puntos muestran la media y las líneas Q1–Q3 de los contenidos y tres pares de semillas. Estos valores son descriptivos: las consultas y pares comparten datos, por lo que no constituyen observaciones independientes.

In [ ]:
figure("07_reduction_stability_dinov2.png")
figure("08_reduction_stability_clip.png")
labels = configs[["encoder", "method", "configuration_id", "label"]]
joined = stability.merge(labels, on=["encoder", "method", "configuration_id"], validate="many_to_one")
table(joined[["encoder", "method", "label", "k", "seed_pairs", "mean", "median", "Q1", "Q3"]])

## 10. Temporal interpretation

La temporalidad se incorpora después del ajuste. Se selecciona la primera secuencia en orden lexicográfico, su tramo sin ambigüedad más largo de índices consecutivos y una ventana central de hasta 30 contenidos. Se excluyen índices nominales duplicados. La regla no consulta las coordenadas y selecciona exactamente los mismos contenidos en DINOv2 y CLIP.

La línea indica únicamente orden nominal entre índices consecutivos inferidos del nombre. No establece timestamps, FPS, velocidad ni movimiento físico. Un recorrido local o un salto en la proyección debe contrastarse con el espacio original y con inspección visual; no confirma leakage.

In [ ]:
example = pd.read_parquet(tables / "temporal_example.parquet")
table(example.groupby("encoder").agg(sequence=("sequence_id", "first"), contents=("content_id", "size"), first_index=("frame_index", "min"), last_index=("frame_index", "max")).reset_index())
figure("09_temporal_overlay_dinov2_pacmap.png")
figure("10_temporal_overlay_clip_pacmap.png")

## 11. Historical split overlay

Cada contenido puede pertenecer históricamente a varios splits debido a copias exactas. Las categorías conservan todas sus pertenencias: train only, val only, test only, train+val y train+test, entre otras si existen. El gráfico audita la partición histórica y no crea una nueva. Colores mezclados o separados no cuantifican por sí solos leakage residual.

In [ ]:
table(pd.read_csv(tables / "historical_memberships.csv"))
figure("11_historical_split_overlay.png")

## 12. Candidate configurations

Regla fijada antes de ejecutar el grid completo: tres semillas válidas y sin colapso/rango deficiente; cinco criterios por configuración (T, C, preservación Jaccard y estabilidad promediados en k=5/10/20, más Spearman). Se conserva el frente no dominado y se selecciona la menor media de rangos, con igual peso por criterio. Spearman aporta un quinto criterio complementario, sin primacía. Los empates se resuelven por estabilidad, preservación, T, Spearman y configuration_id. Se conserva **semilla 0** como referencia, sin buscar la figura más atractiva.

La primera tabla muestra las cuatro referencias: T/C/J/Spearman corresponden al run de semilla 0; estabilidad corresponde a los tres pares de semillas de su configuración. La segunda conserva las alternativas y agregados que determinan la selección. Los rangos solo comparan configuraciones del mismo encoder y método.

In [ ]:
table(references)
table(configs[["encoder", "method", "label", "eligible", "trustworthiness_mean", "continuity_mean", "preservation_mean", "stability_mean", "spearman_mean", "pareto_nondominated", "mean_criterion_rank"]])
display(Markdown("**Trazabilidad por ejecución.** Los identificadores de reducción permiten localizar coordenadas, índices, métricas, conteos de pares y metadata. Los hashes de contenido y rutas privadas no se muestran."))
table(runs[["encoder", "method", "label", "seed", "reduction_space_id"]])

## 13. Limitations

Las métricas evalúan preservación geométrica respecto de cada encoder, no calidad de detección. Las densidades y distancias 2D no son equivalentes a las del espacio original. La selección usa una regla exploratoria, con criterios correlacionados y pesos explícitos; no demuestra una configuración universalmente óptima. El grid es pequeño y no incluye pre-PCA ni ejecuciones 3D. FAISS aproxima vecinos internos de PaCMAP; la evaluación usa rankings completos y vecinos exactos de las coordenadas guardadas.

La temporalidad sigue inferida y no hay timestamps verificados. No se usan clases en las figuras principales, evitando atribuir una sola clase a imágenes con varias anotaciones. Se mantienen conflictos de anotación y pertenencias históricas. No se ejecutaron DBSCAN, OPTICS, HDBSCAN, AMI/ARI, nuevos splits ni YOLO. Bhattacharyya permanece condicionado a una representación distribucional válida.

Las bibliotecas, revisiones y fuentes se registran por ejecución. Los ajustes se realizaron sobre el worktree de implementación y conservan ese commit y sus huellas, sin atribuir retroactivamente los experimentos al commit de cierre. Los datos y outputs permanecen fuera del repositorio público.

## 14. Next step

Definir un protocolo acotado de **DBSCAN, OPTICS y HDBSCAN** que contraste los embeddings L2 originales con estas representaciones candidatas. Elegir escalas de distancia y parámetros propios de cada espacio; no transferir un epsilon numérico entre proyecciones.

Evaluar estabilidad entre semillas/perturbaciones, coherencia visual y temporal con sus límites, fracción de ruido y cobertura. AMI/ARI se calcularán únicamente después de obtener asignaciones reales, documentando cómo se trata el ruido. Preservar content_id y el mapping de ocurrencias; original_split sigue siendo una referencia de auditoría. La futura partición mantendrá íntegros los grupos de alta correlación y medirá el solapamiento residual. Clustering permanece **PLANNED / NEXT**.